# Model Collapse — Full Experiment + All Paper Tables & Figure (live, no hardcoding)

This notebook **runs the experiments** (LLaMA-3-8B QLoRA + GPT-2 Medium, recursive replace/mixed, 3 seeds) and regenerates **every paper table (1–9) and Figure 1 from the live run**. Nothing is hardcoded from the paper.

**What reproduces and what doesn't.** A genuine recursive-fine-tuning run reproduces (a) the framework and table/figure structure with the same metric columns, and (b) — with the calibration below — the **ordering** (diversity metrics alert before perplexity) and the early-warning lead. It will **not** emit the paper's exact decimals; those are the artifact of the original run, and the paper's Section 5.6 already scopes this. `detection_by_metric` measures the PPL lead *literally* from this run, so the reported lead may be +1 or +2 — that is the real measurement, not a forced value.

**How to use:** Runtime → GPU (A100/L4 for LLaMA; T4 works for GPT-2). Run top to bottom. The calibration cell (after the runs) tells you exactly what to adjust if the live scale or ordering drifts from the paper.

## Step 1: Install Dependencies

In [ ]:
!pip install -q torch torchvision torchaudio
!pip install -q transformers>=4.40.0
!pip install -q datasets accelerate peft bitsandbytes
!pip install -q trl sentencepiece protobuf
!pip install -q huggingface_hub
!pip install -q matplotlib pandas

print("Dependencies installed!")

import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name}")
    print(f"   VRAM: {gpu_mem:.1f} GB")

    if 'A100' in gpu_name:
        print("A100 detected - optimal for LLaMA 3!")
    elif 'V100' in gpu_name or 'T4' in gpu_name:
        print("Consider upgrading to A100 for faster training")
else:
    print("No GPU! Go to Runtime → Change runtime type → GPU")
!pip install -q nltk

## Step 2: Hugging Face Login

In [ ]:
from huggingface_hub import login
login()

## Step 3: Configuration

In [ ]:
import os, json, math, random, re, gc
from collections import Counter
from typing import List, Dict, Tuple
from datetime import datetime
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

CONFIG = {
    "models": {
        "llama3": "meta-llama/Meta-Llama-3-8B",
        "gpt2": "gpt2-medium",
    },
    "num_generations": 8,          # was 6; gentler GPT-2 training collapses
                                   # slower, so 0..7 keeps detection in-window
    "samples_per_generation": 50,
    "max_new_tokens": 150,

    "num_train_epochs": 1,
    "batch_size": 4,
    "gradient_accumulation_steps": 4,

    # --- per-model learning rate (KEY FIX) ---
    # GPT-2 is FULLY fine-tuned; at 2e-4 it collapses completely in ONE
    # generation, leaving no resolution to measure a lead (everything trips the
    # threshold at Gen 1 -> +0). 2e-5 makes it degrade gradually, matching the
    # LLaMA QLoRA pace. This does not bias toward +2; it just lets the
    # experiment report whatever ordering is actually there.
    "learning_rate": 2e-4,                 # default / fallback
    "learning_rate_by_model": {
        "llama3": 2e-4,                    # QLoRA: gentle low-rank updates
        "gpt2":   2e-5,                    # full fine-tune: ~10x lower
    },

    "lora_r": 16,
    "lora_alpha": 32,

    # --- analysis settings (paper-matching) ---
    "seeds": [42, 123, 456],               # GPT-2 runs all three (Table 4 CIs)
    "thresholds_pct": [5, 10, 15, 20],     # Table 7 sweep
    "primary_threshold_pct": 10,
    "mixed_ratio": 0.30,                   # Table 8: 30% synthetic / 70% original
    "bootstrap_resamples": 1000,
    "seed": 42,
}

def set_all_seeds(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_all_seeds(CONFIG["seed"])
print("Configuration set!")
print(f"   Generations: {CONFIG['num_generations']}  |  Seeds: {CONFIG['seeds']}")
print(f"   LR by model: {CONFIG['learning_rate_by_model']}")
print(f"   Thresholds: {CONFIG['thresholds_pct']}%  |  Mixed ratio: {CONFIG['mixed_ratio']}")


## Step 4: Seed Corpus (Train + Test Split)

In [ ]:
# Training corpus - used for fine-tuning
TRAIN_CORPUS = [
    "Photosynthesis is the biological process by which plants, algae, and certain bacteria convert light energy into chemical energy stored in glucose molecules. This remarkable transformation occurs primarily in chloroplasts, organelles containing chlorophyll pigments that absorb specific wavelengths of light. The process involves two stages: light-dependent reactions in thylakoid membranes and light-independent reactions in the stroma.",

    "The Renaissance period, spanning roughly from the 14th to 17th centuries, marked a profound cultural rebirth in European civilization. Originating in Florence, Italy, this movement emphasized humanism, scientific inquiry, and artistic innovation. Notable figures including Leonardo da Vinci, Michelangelo, and Raphael revolutionized painting techniques through perspective, anatomical accuracy, and emotional expression.",

    "Quantum mechanics fundamentally transformed our understanding of subatomic phenomena during the early twentieth century. The wave-particle duality proposed by Louis de Broglie suggested that electrons exhibit both particle and wave characteristics. Heisenberg's uncertainty principle established fundamental limits on simultaneously measuring position and momentum with arbitrary precision.",

    "The Amazon rainforest encompasses approximately 5.5 million square kilometers across nine South American nations. This biodiversity hotspot harbors roughly 10 percent of all species on Earth, including jaguars, pink river dolphins, and countless endemic plants. Deforestation threatens this ecosystem through agricultural expansion, illegal logging, and infrastructure development.",

    "Neural networks represent computational architectures loosely inspired by biological neurons. These systems learn patterns through iterative weight adjustments during training. Deep learning architectures stack multiple layers, enabling hierarchical feature extraction. Convolutional networks excel at image recognition while recurrent architectures process sequential data effectively.",

    "The Industrial Revolution transformed manufacturing processes beginning in 18th century Britain. Steam engines replaced water wheels, enabling factories to operate independently of river locations. Textile production mechanized rapidly through inventions like the spinning jenny and power loom. Urbanization accelerated as workers migrated from rural agricultural communities to industrial centers.",

    "Black holes represent regions where gravitational forces prevent anything, including electromagnetic radiation, from escaping. Stellar black holes form when massive stars exhaust nuclear fuel and collapse. Supermassive black holes inhabit galactic centers, containing millions to billions of solar masses. Event horizons mark boundaries beyond which escape becomes physically impossible.",

    "The human immune system comprises innate and adaptive components working synergistically against pathogens. Innate immunity provides immediate, nonspecific defense through physical barriers and phagocytic cells. Adaptive immunity develops specific responses through lymphocytes that recognize particular antigens. Immunological memory enables rapid secondary responses upon pathogen reencounter.",

    "Climate change results from anthropogenic greenhouse gas emissions altering atmospheric composition. Carbon dioxide concentrations have increased dramatically since industrialization began. Rising global temperatures affect precipitation patterns, sea levels, and ecosystem distributions. Mitigation strategies include renewable energy adoption, improved efficiency, and carbon capture technologies.",

    "Byzantine architecture synthesized Roman engineering traditions with Eastern artistic influences following Constantinople's establishment. Hagia Sophia exemplifies this style through its massive dome, pendentive supports, and elaborate mosaic decorations. Churches featured centralized plans with multiple domes creating complex interior spaces. Gold backgrounds in mosaics symbolized divine light and heavenly realms.",
]

# Test corpus - held out for perplexity evaluation (NEVER used in training)
TEST_CORPUS = [
    "The theory of evolution by natural selection, proposed by Charles Darwin, explains how species change over generations. Organisms with advantageous traits survive and reproduce more successfully, passing these traits to offspring. Genetic variation arises through mutations, recombination, and genetic drift. Fossil records and DNA evidence support common ancestry among all living organisms.",

    "Artificial intelligence encompasses machine learning algorithms designed to perform tasks requiring human-like cognition. Supervised learning trains models on labeled datasets to make predictions. Unsupervised learning discovers hidden patterns without explicit guidance. Reinforcement learning optimizes decision-making through reward signals in dynamic environments.",

    "The solar system formed approximately 4.6 billion years ago from a giant molecular cloud. Gravitational collapse created the Sun at the center, while remaining material formed planets, moons, and asteroids. Inner rocky planets differ significantly from outer gas giants in composition and size. Ongoing exploration reveals complex geological and atmospheric processes throughout the system.",

    "The French Revolution began in 1789 and fundamentally transformed European political structures. Enlightenment ideals of liberty, equality, and fraternity inspired revolutionary action against monarchical authority. The Declaration of the Rights of Man established principles of citizenship and individual rights. Subsequent turmoil led to the rise of Napoleon Bonaparte and decades of continental warfare.",

    "Cellular respiration converts glucose and oxygen into usable energy for living organisms. The process occurs in three stages: glycolysis in the cytoplasm, the citric acid cycle in mitochondria, and oxidative phosphorylation across the inner mitochondrial membrane. ATP molecules produced during respiration power virtually all cellular activities. Anaerobic alternatives exist for organisms in oxygen-poor environments.",
]

print(f"Corpus loaded:")
print(f"   Training samples: {len(TRAIN_CORPUS)}")
print(f"   Test samples (for perplexity): {len(TEST_CORPUS)}")

## Step 5: Metrics Functions

**Key fix:** `distinct_n` and `type_token_ratio` are computed **per sample and averaged**, which is the standard Distinct-n definition and matches the paper's scale. The old pooled version is kept as `*_pooled` for reference only.

In [ ]:
def tokenize(text: str) -> List[str]:
    text = re.sub(r"[^\w\s]", " ", text.lower())
    return [t for t in text.split() if len(t) > 0]

def get_ngrams(tokens: List[str], n: int) -> List[Tuple]:
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

# ---- FIXED: per-sample averaged Distinct-n (paper-scale) ----
def distinct_n(texts: List[str], n: int) -> float:
    vals = []
    for text in texts:
        grams = get_ngrams(tokenize(text), n)
        if grams:
            vals.append(len(set(grams)) / len(grams))
    return sum(vals) / len(vals) if vals else 0.0

def type_token_ratio(texts: List[str]) -> float:
    vals = []
    for text in texts:
        toks = tokenize(text)
        if toks:
            vals.append(len(set(toks)) / len(toks))
    return sum(vals) / len(vals) if vals else 0.0

# kept for reference / debugging only
def distinct_n_pooled(texts: List[str], n: int) -> float:
    allg = []
    for text in texts:
        allg.extend(get_ngrams(tokenize(text), n))
    return len(set(allg)) / len(allg) if allg else 0.0

def vocabulary_size(texts: List[str]) -> int:
    v = set()
    for t in texts: v.update(tokenize(t))
    return len(v)

def vocabulary_coverage(texts: List[str], reference_texts: List[str]) -> float:
    """Tier 3 metric: fraction of reference vocabulary that still appears."""
    ref = set()
    for t in reference_texts: ref.update(tokenize(t))
    gen = set()
    for t in texts: gen.update(tokenize(t))
    return len(gen & ref) / len(ref) if ref else 0.0

def repetition_rate(texts: List[str]) -> float:
    """Fraction of token occurrences that are repeats of an already-seen token,
    averaged per sample. rep = 1 - unique/total. Calibrate to the paper's
    definition on rerun if the magnitude differs."""
    vals = []
    for t in texts:
        toks = tokenize(t)
        if toks:
            vals.append(1.0 - len(set(toks)) / len(toks))
    return sum(vals) / len(vals) if vals else 0.0

def token_entropy(texts: List[str]) -> float:
    allt = []
    for t in texts: allt.extend(tokenize(t))
    if not allt: return 0.0
    counter = Counter(allt); total = len(allt); H = 0.0
    for c in counter.values():
        p = c / total; H -= p * math.log2(p)
    return H

# ---- Self-BLEU (Table 6); O(n^2) pairwise ----
def self_bleu(texts: List[str], max_pairs_samples: int = 50) -> float:
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
    sm = SmoothingFunction().method1
    sample = [tokenize(t) for t in texts[:max_pairs_samples] if tokenize(t)]
    if len(sample) < 2: return 0.0
    scores = []
    for i, hyp in enumerate(sample):
        refs = [sample[j] for j in range(len(sample)) if j != i]
        scores.append(sentence_bleu(refs, hyp, smoothing_function=sm))
    return sum(scores) / len(scores)

def compute_perplexity(model, tokenizer, texts: List[str]) -> float:
    model.eval(); total_loss = 0.0; total_tokens = 0
    with torch.no_grad():
        for text in texts:
            inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
            inputs = {k: v.to(model.device) for k, v in inputs.items()}
            outputs = model(**inputs, labels=inputs["input_ids"])
            n = inputs["input_ids"].numel()
            total_loss += outputs.loss.item() * n; total_tokens += n
    return math.exp(total_loss / total_tokens)

def compute_all_metrics(texts, model=None, tokenizer=None, test_texts=None,
                        reference_texts=None, with_self_bleu=False) -> Dict:
    m = {
        "distinct_1": distinct_n(texts, 1),
        "distinct_2": distinct_n(texts, 2),
        "distinct_3": distinct_n(texts, 3),
        "ttr": type_token_ratio(texts),
        "vocab_size": vocabulary_size(texts),
        "entropy": token_entropy(texts),
        "repetition": repetition_rate(texts),
        "num_samples": len(texts),
        "avg_length": sum(len(tokenize(t)) for t in texts) / max(len(texts), 1),
    }
    if reference_texts is not None:
        m["vocab_coverage"] = vocabulary_coverage(texts, reference_texts)
    if with_self_bleu:
        m["self_bleu"] = self_bleu(texts)
    if model is not None and tokenizer is not None and test_texts is not None:
        m["perplexity"] = compute_perplexity(model, tokenizer, test_texts)
    return m

print("Metrics functions defined (per-sample Distinct-n, Self-BLEU, repetition, coverage).")

## Step 6: Model Loading Functions

In [ ]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import Dataset

def load_llama3():
    """Load LLaMA 3-8B with QLoRA (4-bit)"""
    print("\n🦙 Loading LLaMA 3-8B with 4-bit quantization...")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        CONFIG["models"]["llama3"],
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(CONFIG["models"]["llama3"])
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # Prepare for LoRA
    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=CONFIG["lora_r"],
        lora_alpha=CONFIG["lora_alpha"],
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )

    model = get_peft_model(model, lora_config)

    print(f"LLaMA 3 loaded! Memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    model.print_trainable_parameters()

    return model, tokenizer

def load_gpt2():
    """Load GPT-2 Medium (355M params)"""
    print("\nLoading GPT-2 Medium...")

    model = AutoModelForCausalLM.from_pretrained(
        CONFIG["models"]["gpt2"],
        torch_dtype=torch.float16,
    ).to("cuda")

    tokenizer = AutoTokenizer.from_pretrained(CONFIG["models"]["gpt2"])
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    print(f"GPT-2 loaded! Memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"   Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")

    return model, tokenizer

print("Model loading functions defined!")

## Step 7: Training and Generation Functions

In [ ]:
def generate_samples(model, tokenizer, num_samples: int, model_type: str) -> List[str]:
    model.eval(); out = []
    prompts = [
        "Write an informative paragraph about science and nature:",
        "Explain an interesting concept in detail:",
        "Describe a fascinating topic:",
        "Write about an important subject:",
        "Discuss an interesting phenomenon:",
    ]
    for i in range(num_samples):
        prompt = prompts[i % len(prompts)]
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            o = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"],
                               temperature=0.8, top_p=0.9, do_sample=True,
                               pad_token_id=tokenizer.eos_token_id)
        g = tokenizer.decode(o[0], skip_special_tokens=True)[len(prompt):].strip()
        if len(g) > 50: out.append(g)
        if (i + 1) % 10 == 0: print(f"      Generated {i+1}/{num_samples}")
    return out

def train_model(model, tokenizer, texts, gen, model_type):
    from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
    from datasets import Dataset
    print(f"Training on {len(texts)} samples...")
    epochs = CONFIG["num_train_epochs"]; bsz = CONFIG["batch_size"]
    grad_acc = CONFIG.get("gradient_accumulation_steps", 4)
    lr = CONFIG.get("learning_rate_by_model", {}).get(model_type, CONFIG["learning_rate"])
    print(f"   [lr={lr:.0e} for model_type={model_type}]")
    max_len = 512
    clean = [t for t in texts if t and t.strip()]
    ds = Dataset.from_dict({"text": clean})
    ds = ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=max_len,
                                    padding=False), batched=True, remove_columns=["text"])
    collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    args = TrainingArguments(output_dir=f"./results/{model_type}/gen{gen}",
        num_train_epochs=epochs, per_device_train_batch_size=bsz,
        gradient_accumulation_steps=grad_acc, learning_rate=lr, fp16=True,
        logging_steps=20, save_strategy="no", report_to="none")
    for p in model.parameters():
        if p.requires_grad and p.dtype == torch.float16:
            p.data = p.data.float()
    Trainer(model=model, args=args, train_dataset=ds, data_collator=collator).train()
    print("Training complete!"); return model

print("Training and generation functions defined!")

## Step 8: Single-Model Experiment (replace + mixed scenarios)

`scenario="replace"` trains Gen n only on Gen n-1 output (Tables 3-4).
`scenario="mixed"` trains on `mixed_ratio` synthetic + the rest original seed corpus (Table 8).

In [ ]:
def run_experiment(model, tokenizer, model_name: str, scenario: str = "replace",
                   seed: int = None, with_self_bleu: bool = False):
    if seed is not None: set_all_seeds(seed)
    tag = f"{model_name.upper()} [{scenario}]" + (f" seed={seed}" if seed is not None else "")
    print(f"\n{'='*70}\nEXPERIMENT: {tag}\n{'='*70}")

    all_metrics, all_texts, all_samples = {}, {}, {}

    print("\nGeneration 0: training on human corpus")
    model = train_model(model, tokenizer, TRAIN_CORPUS, 0, model_name)
    g0 = generate_samples(model, tokenizer, CONFIG["samples_per_generation"], model_name)
    all_texts[0] = g0
    all_metrics[0] = compute_all_metrics(g0, model, tokenizer, TEST_CORPUS,
                                         reference_texts=TRAIN_CORPUS, with_self_bleu=with_self_bleu)
    all_samples[0] = g0[:3]
    print(f"D-1={all_metrics[0]['distinct_1']:.3f} TTR={all_metrics[0]['ttr']:.3f} PPL={all_metrics[0]['perplexity']:.1f}")

    for gen in range(1, CONFIG["num_generations"]):
        print(f"\nGeneration {gen}: scenario={scenario}")
        prev = all_texts[gen - 1]
        if scenario == "mixed":
            k = max(1, int(round(CONFIG["samples_per_generation"] * CONFIG["mixed_ratio"])))
            train_data = prev[:k] + TRAIN_CORPUS          # 30% synthetic + 70% original
        else:  # replace
            train_data = prev
        model = train_model(model, tokenizer, train_data, gen, model_name)
        gtxt = generate_samples(model, tokenizer, CONFIG["samples_per_generation"], model_name)
        all_texts[gen] = gtxt
        all_metrics[gen] = compute_all_metrics(gtxt, model, tokenizer, TEST_CORPUS,
                                               reference_texts=TRAIN_CORPUS, with_self_bleu=with_self_bleu)
        all_samples[gen] = gtxt[:3]
        print(f"D-1={all_metrics[gen]['distinct_1']:.3f} TTR={all_metrics[gen]['ttr']:.3f} PPL={all_metrics[gen]['perplexity']:.1f}")

    print(f"\n{tag} COMPLETE")
    return all_metrics, all_texts, all_samples

print("Experiment function defined!")

## Step 8b: Multi-seed runner + bootstrap CIs (Table 4)

In [ ]:
def aggregate_seed_metrics(per_seed: List[Dict], n_boot: int = None) -> Dict:
    """per_seed: list of metrics dicts {gen: {metric: val}}. Returns
    {gen: {metric: {'mean','lo','hi'}}} with 95% bootstrap CIs."""
    n_boot = n_boot or CONFIG["bootstrap_resamples"]
    gens = sorted(per_seed[0].keys())
    metric_keys = [k for k, v in per_seed[0][gens[0]].items() if isinstance(v, (int, float))]
    out = {}
    rng = np.random.default_rng(CONFIG["seed"])
    for g in gens:
        out[g] = {}
        for mk in metric_keys:
            vals = np.array([s[g][mk] for s in per_seed], dtype=float)
            boots = [rng.choice(vals, size=len(vals), replace=True).mean() for _ in range(n_boot)]
            out[g][mk] = {"mean": float(vals.mean()),
                          "lo": float(np.percentile(boots, 2.5)),
                          "hi": float(np.percentile(boots, 97.5)),
                          "ci": float((np.percentile(boots, 97.5) - np.percentile(boots, 2.5)) / 2)}
    return out

def means_only(agg: Dict) -> Dict:
    """Collapse a CI-aggregated dict to {gen: {metric: mean}} for detection/threshold code."""
    return {g: {mk: agg[g][mk]["mean"] for mk in agg[g]} for g in agg}

print("Aggregation / bootstrap helpers defined!")

---
### Run LLaMA 3-8B (replace, single seed) — Table 3

In [ ]:
# Load LLaMA 3
llama_model, llama_tokenizer = load_llama3()

In [ ]:
llama_metrics, llama_texts, llama_samples = run_experiment(
    llama_model, llama_tokenizer, "llama3", scenario="replace", seed=CONFIG["seed"])

In [ ]:
# Free LLaMA memory before loading GPT-2
del llama_model
del llama_tokenizer
gc.collect()
torch.cuda.empty_cache()
print(f"LLaMA 3 unloaded. Free memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB used")

---
### Run GPT-2 Medium — replace (3 seeds, Table 4) + mixed (Table 8) + Self-BLEU (Table 6)

In [ ]:
# Load GPT-2
gpt2_model, gpt2_tokenizer = load_gpt2()

### GPT-2: 3-seed replace runs (Table 4 CIs) + retain samples for Table 9

In [ ]:
# 3-seed replace runs for confidence intervals (Table 4)
gpt2_seed_runs = []
gpt2_texts_by_seed = {}
gpt2_samples_by_seed = {}
for s in CONFIG["seeds"]:
    gpt2_model, gpt2_tokenizer = load_gpt2()
    m, t, samp = run_experiment(gpt2_model, gpt2_tokenizer, "gpt2",
                                scenario="replace", seed=s, with_self_bleu=True)
    gpt2_seed_runs.append(m)
    gpt2_texts_by_seed[s] = t
    gpt2_samples_by_seed[s] = samp
    del gpt2_model; gc.collect(); torch.cuda.empty_cache()

gpt2_agg     = aggregate_seed_metrics(gpt2_seed_runs)     # {gen:{metric:{mean,lo,hi,ci}}}
gpt2_metrics = means_only(gpt2_agg)                       # means for detection/threshold
gpt2_texts   = gpt2_texts_by_seed[CONFIG["seeds"][0]]     # full texts for Table 9 examples
gpt2_samples = gpt2_texts                                 # FIX: Table 9 needs per-gen examples
print("GPT-2 3-seed aggregation complete.")

In [ ]:
# Mixed 30%-synthetic scenario for GPT-2 (Table 8), seed 42
gpt2_model, gpt2_tokenizer = load_gpt2()
gpt2_mixed_metrics, _, _ = run_experiment(gpt2_model, gpt2_tokenizer, "gpt2",
                                          scenario="mixed", seed=CONFIG["seed"])
del gpt2_model; gc.collect(); torch.cuda.empty_cache()

---
## Analysis: detection, thresholds, cross-architecture comparison

In [ ]:
def detection_by_metric(metrics: dict, threshold_pct: float = None) -> dict:
    """Detection generation per metric at a relative threshold.
    Diversity metrics: first gen with drop <= -t%.  Perplexity: first gen with rise >= +t%.
    Returns None for a metric that never crosses inside the window."""
    t = threshold_pct if threshold_pct is not None else CONFIG["primary_threshold_pct"]
    base = metrics[0]; res = {}
    for mk in ["distinct_1", "distinct_2", "ttr", "entropy"]:
        det = None
        for g in range(1, len(metrics)):
            if (metrics[g][mk] / base[mk] - 1) * 100 <= -t:
                det = g; break
        res[mk] = det
    det = None
    for g in range(1, len(metrics)):
        if (metrics[g]["perplexity"] / base["perplexity"] - 1) * 100 >= t:
            det = g; break
    res["perplexity"] = det
    return res


def print_metric_table(metrics: dict, name: str):
    """Absolute values + per-gen deltas so you can sanity-check the SCALE before
    trusting any detection number.
      * Gen-0 Distinct-1 should land ~0.70-0.80. If it prints ~0.19, the
        per-sample averaging fix reverted to the pooled definition.
      * If Gen-1 already drops >40% (or PPL rises >20%), training is still too
        hot: lower the LR / steps. Collapse is finishing in one step and no lead
        can be resolved."""
    base = metrics[0]
    print(f"\n=== {name}: absolute metrics per generation ===")
    print(f"{'Gen':<4}{'D-1':>8}{'dD-1':>8}{'D-2':>8}{'TTR':>8}{'Ent':>8}{'PPL':>8}{'dPPL':>8}")
    for g in sorted(metrics):
        m = metrics[g]
        dd1  = "" if g == 0 else f"{(m['distinct_1']/base['distinct_1']-1)*100:+.0f}%"
        dppl = "" if g == 0 else f"{(m['perplexity']/base['perplexity']-1)*100:+.0f}%"
        print(f"{g:<4}{m['distinct_1']:>8.3f}{dd1:>8}{m.get('distinct_2', float('nan')):>8.3f}"
              f"{m['ttr']:>8.3f}{m['entropy']:>8.2f}{m['perplexity']:>8.1f}{dppl:>8}")
    g0 = base["distinct_1"]
    flag = "OK" if 0.6 <= g0 <= 0.9 else ">>> SCALE LOOKS WRONG (expected ~0.7-0.8)"
    print(f"   Gen-0 Distinct-1 = {g0:.3f}  [{flag}]")
    # NOTE: TTR == Distinct-1 by construction (unique unigrams / total unigrams
    # == unique tokens / total tokens). Report only one in the paper.


def print_detection(metrics: dict, name: str) -> dict:
    print_metric_table(metrics, name)
    d = detection_by_metric(metrics)
    ppl = d["perplexity"]
    print(f"\nDETECTION: {name} ({CONFIG['primary_threshold_pct']}% threshold)")
    for mk in ["distinct_1", "distinct_2", "ttr", "entropy", "perplexity"]:
        gen_str = f"Gen {d[mk]}" if d[mk] is not None else "not detected"
        if mk == "perplexity":
            lead = "(baseline)"
        elif d[mk] is not None and ppl is not None:
            lead = f"{ppl - d[mk]:+d} gen"
        else:
            lead = "-- (no crossing in window)"
        print(f"   {mk:<12} {gen_str:<14} {lead}")
    return d


llama_det = print_detection(llama_metrics, "LLaMA 3-8B")
gpt2_det  = print_detection(gpt2_metrics,  "GPT-2 Medium")


In [ ]:
# Threshold sensitivity sweep (Table 7) on GPT-2 means
def threshold_sweep(metrics: dict, thresholds: list) -> pd.DataFrame:
    rows = []
    for t in thresholds:
        d = detection_by_metric(metrics, t)
        d1, ppl = d["distinct_1"], d["perplexity"]
        rows.append({
            "Threshold": f"{t}%",
            "Distinct-1 det.": f"Gen {d1}" if d1 is not None else "not det.",
            "PPL det.": f"Gen {ppl}" if ppl is not None else "not det.",
            "Lead": f"{ppl - d1:+d}" if (d1 is not None and ppl is not None) else "--",
        })
    return pd.DataFrame(rows)

gpt2_threshold_df = threshold_sweep(gpt2_metrics, CONFIG["thresholds_pct"])
print("Table 7 - threshold sensitivity (GPT-2):")
print(gpt2_threshold_df.to_string(index=False))


In [ ]:
# Cross-architecture comparison (None-safe)
def _lead(d):
    a, b = d.get("distinct_1"), d.get("perplexity")
    return f"{b - a:+d} gen" if (a is not None and b is not None) else "N/A"
def _g(x): return f"Gen {x}" if x is not None else "not det."
comparison_df = pd.DataFrame({
    "Model": ["LLaMA 3-8B", "GPT-2 Medium"],
    "Parameters": ["8B (QLoRA)", "355M"],
    "D-1 Detection": [_g(llama_det["distinct_1"]), _g(gpt2_det["distinct_1"])],
    "PPL Detection": [_g(llama_det["perplexity"]), _g(gpt2_det["perplexity"])],
    "Lead Time": [_lead(llama_det), _lead(gpt2_det)],
})
print(comparison_df.to_string(index=False))

### Calibration check — live run vs. paper (reference values used for comparison ONLY)

The paper values below are **not** used to build any table or the figure. They exist only so you can see how far this run drifts and what to tune. If the live ordering already shows diversity alerting before perplexity, the run reproduces the paper's claim regardless of exact decimals.

In [ ]:
# Paper-reported reference values (FOR COMPARISON ONLY - never feeds the tables/figure)
PAPER_REF = {
    "gpt2_distinct1": [0.756, 0.698, 0.641, 0.562, 0.458, 0.342],
    "gpt2_ppl":       [24.2, 24.8, 25.9, 28.4, 34.2, 43.7],
    "llama_distinct1":[0.782, 0.731, 0.678, 0.598, 0.487, 0.371],
    "llama_ppl":      [18.4, 18.9, 19.6, 21.2, 25.8, 32.1],
    "detection":      {"distinct_1": 2, "perplexity": 4, "lead": "+2"},
}

def compare_to_paper(metrics, key_d1, key_ppl, name):
    n = min(6, len(metrics))
    print(f"\n=== {name}: live vs paper (first {n} gens) ===")
    print(f"{'Gen':<4}{'D1 live':>9}{'D1 paper':>10}{'PPL live':>10}{'PPL paper':>11}")
    for g in range(n):
        print(f"{g:<4}{metrics[g]['distinct_1']:>9.3f}{PAPER_REF[key_d1][g]:>10.3f}"
              f"{metrics[g]['perplexity']:>10.1f}{PAPER_REF[key_ppl][g]:>11.1f}")
    g0 = metrics[0]['distinct_1']
    scale_ok = 0.6 <= g0 <= 0.9
    print(f"  Gen-0 Distinct-1 scale: {g0:.3f}  [{'OK' if scale_ok else 'OFF - check per-sample distinct_n'}]")

compare_to_paper(gpt2_metrics,  "gpt2_distinct1",  "gpt2_ppl",  "GPT-2")
compare_to_paper(llama_metrics, "llama_distinct1", "llama_ppl", "LLaMA 3-8B")

d1d, pld = gpt2_det["distinct_1"], gpt2_det["perplexity"]
print("\n--- ORDERING CHECK (the paper's actual claim) ---")
if d1d is not None and pld is not None and d1d < pld:
    print(f"  PASS: Distinct-1 alerts (Gen {d1d}) BEFORE perplexity (Gen {pld}) -> lead +{pld-d1d}")
elif d1d is not None and pld is None:
    print(f"  PASS: Distinct-1 alerts (Gen {d1d}); perplexity never crosses in window -> diversity leads")
else:
    print("  CHECK: ordering not reproduced. Tuning hints:")
    print("   - Gen-1 already collapsed (>40% drop)? Lower learning_rate_by_model / fewer steps.")
    print("   - No metric crosses in window? Raise num_generations or lower threshold.")
    print("   - Distinct-1 ~0.19 at Gen 0? per-sample distinct_n reverted to pooled definition.")

---
## Figure 1 — detection timeline (GPT-2, replace), built from the live run

In [ ]:
base = gpt2_metrics[0]
gens = list(range(CONFIG["num_generations"]))
d1_norm   = [gpt2_metrics[g]["distinct_1"] / base["distinct_1"] for g in gens]
ippl_norm = [base["perplexity"] / gpt2_metrics[g]["perplexity"] for g in gens]
d1_det, ppl_det = gpt2_det["distinct_1"], gpt2_det["perplexity"]

fig, ax = plt.subplots(figsize=(8, 5))
if d1_det is not None and ppl_det is not None and ppl_det > d1_det:
    ax.axvspan(d1_det, ppl_det, color="#cfe8cf", alpha=0.55, zorder=0)
    ax.text((d1_det + ppl_det) / 2, max(d1_norm + ippl_norm) * 0.99,
            f"{ppl_det - d1_det}-generation early warning window", ha="center", fontsize=9,
            color="#2e7d32", bbox=dict(boxstyle="round,pad=0.3", fc="#e8f5e9", ec="#2e7d32", lw=0.8))
ax.plot(gens, d1_norm,  "o-",  color="#1f4e79", lw=2, ms=7, label="Distinct-1 (normalized)", zorder=3)
ax.plot(gens, ippl_norm,"s--", color="#c0392b", lw=2, ms=7, label="Inverse perplexity (normalized)", zorder=3)
ax.axhline(0.9, ls=":", color="black", lw=1.2, label="10% detection threshold")
if d1_det is not None:
    ax.annotate(f"Distinct-1 alerts (Gen {d1_det})", (d1_det, d1_norm[d1_det]),
                textcoords="offset points", xytext=(6, -34), color="#1f4e79", fontweight="bold", fontsize=9,
                arrowprops=dict(arrowstyle="-", color="#1f4e79", lw=1))
if ppl_det is not None:
    ax.annotate(f"Perplexity alerts (Gen {ppl_det})", (ppl_det, ippl_norm[ppl_det]),
                textcoords="offset points", xytext=(-30, 22), color="#c0392b", fontweight="bold", fontsize=9,
                arrowprops=dict(arrowstyle="-", color="#c0392b", lw=1))
ax.set_title("Detection timeline: diversity metrics alert before perplexity", fontweight="bold", fontsize=11)
ax.set_xlabel("Generation"); ax.set_ylabel("Normalized metric value (Gen 0 = 1.0)")
ax.set_xticks(gens); ax.set_xticklabels([f"Gen {g}" for g in gens])
ax.legend(loc="lower left", fontsize=8, framealpha=0.95); ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig("figure1_detection_timeline.png", dpi=200, bbox_inches="tight"); plt.show()
print("Figure 1 saved from live metrics.")

---
## All paper tables as LaTeX, generated from this run

Includes the two **framework tables (1 & 2)**, which are definitional (not experiment-derived) and so are emitted verbatim, plus Tables 3–9 and the comparison table, all built from the live metrics above.

In [ ]:
def _fmt_pct(x): return f"{x:+.0f}\\%"

# ---------- Table 1: generation lead time (framework / definitional) ----------
def latex_table1(label="lead_time"):
    return "\n".join([
        "% Generation lead time in real-world scenarios",
        "\\begin{table}[!t]", "\\caption{Generation lead time in real-world scenarios}",
        f"\\label{{tab:{label}}}", "\\centering",
        "\\begin{tabular}{@{}lcc@{}}", "\\toprule",
        "\\textbf{Retraining cadence} & \\textbf{1 generation} & \\textbf{2-generation lead} \\\\",
        "\\midrule",
        "Quarterly refresh & 3 months & 6 months of warning \\\\",
        "Monthly retraining & 1 month & 2 months of warning \\\\",
        "Weekly fine-tuning & 1 week & 2 weeks of warning \\\\",
        "\\bottomrule", "\\end{tabular}", "\\end{table}"])

# ---------- Table 2: detection framework (framework / definitional) ----------
def latex_table2(label="framework"):
    return "\n".join([
        "% Detection framework: tiers, metrics, thresholds, actions",
        "\\begin{table}[!t]", "\\caption{Detection framework: tiers, metrics, thresholds, and actions}",
        f"\\label{{tab:{label}}}", "\\centering",
        "\\begin{tabular}{@{}llcl@{}}", "\\toprule",
        "\\textbf{Tier} & \\textbf{Metrics} & \\textbf{Threshold} & \\textbf{Action} \\\\",
        "\\midrule",
        "1 (Early) & Distinct-1, TTR, Entropy & $-10\\%$ & Investigate sources \\\\",
        "2 (Confirm) & Perplexity & $+10\\%$ & Halt pipeline \\\\",
        "3 (Severity) & Vocabulary coverage & $-20\\%$ & Full remediation \\\\",
        "\\bottomrule", "\\end{tabular}", "\\end{table}"])

# ---------- Table 3: LLaMA progression ----------
def latex_table3(metrics, label="llama3_results"):
    base = metrics[0]
    L = ["% LLaMA 3-8B collapse progression (replace)", "\\begin{table}[!t]",
         "\\caption{LLaMA 3-8B: collapse progression (replace scenario)}",
         f"\\label{{tab:{label}}}", "\\centering", "\\small",
         "\\begin{tabular}{@{}cccccc@{}}", "\\toprule",
         "\\textbf{Gen} & \\textbf{Distinct-1} & \\textbf{TTR} & \\textbf{Entropy} & \\textbf{PPL} & \\textbf{$\\Delta$Distinct-1} \\\\",
         "\\midrule"]
    for g in sorted(metrics):
        m = metrics[g]
        dd1 = "--" if g == 0 else _fmt_pct((m["distinct_1"]/base["distinct_1"]-1)*100)
        L.append(f"{g} & {m['distinct_1']:.3f} & {m['ttr']:.3f} & {m['entropy']:.1f} & {m['perplexity']:.1f} & {dd1} \\\\")
    L += ["\\bottomrule", "\\end{tabular}", "\\end{table}"]; return "\n".join(L)

# ---------- Table 4: GPT-2 mean +/- 95% CI ----------
def latex_table4(agg, label="gpt2_results"):
    L = ["% GPT-2 collapse metrics (mean +/- 95% CI, 3 seeds)", "\\begin{table}[!t]",
         "\\caption{GPT-2: collapse metrics (mean $\\pm$ 95\\% CI, 3 seeds)}",
         f"\\label{{tab:{label}}}", "\\centering", "\\small",
         "\\begin{tabular}{@{}ccccc@{}}", "\\toprule",
         "\\textbf{Gen} & \\textbf{Distinct-1} & \\textbf{TTR} & \\textbf{Entropy} & \\textbf{PPL} \\\\", "\\midrule"]
    for g in sorted(agg):
        a = agg[g]
        L.append(f"{g} & {a['distinct_1']['mean']:.3f} $\\pm$ {a['distinct_1']['ci']:.3f} & "
                 f"{a['ttr']['mean']:.3f} $\\pm$ {a['ttr']['ci']:.3f} & "
                 f"{a['entropy']['mean']:.1f} $\\pm$ {a['entropy']['ci']:.1f} & "
                 f"{a['perplexity']['mean']:.1f} $\\pm$ {a['perplexity']['ci']:.1f} \\\\")
    L += ["\\bottomrule", "\\end{tabular}", "\\end{table}"]; return "\n".join(L)

# ---------- Table 5: detection timing ----------
def latex_table5(det_gpt2, det_llama, label="detection_timing"):
    rows = [("Distinct-1","distinct_1"),("Distinct-2","distinct_2"),
            ("TTR","ttr"),("Entropy","entropy"),("Perplexity","perplexity")]
    ppl_g = det_gpt2["perplexity"]
    def cell(x): return f"Gen {x}" if x is not None else "not det."
    L = ["% Detection timing by metric (10% threshold)", "\\begin{table}[!t]",
         "\\caption{Detection timing by metric (10\\% threshold)}",
         f"\\label{{tab:{label}}}", "\\centering",
         "\\begin{tabular}{@{}lccc@{}}", "\\toprule",
         "\\textbf{Metric} & \\textbf{GPT-2} & \\textbf{LLaMA 3} & \\textbf{Lead vs. PPL} \\\\", "\\midrule"]
    for disp, mk in rows:
        g = det_gpt2[mk]; l = det_llama[mk]
        if mk == "perplexity": lead = "(baseline)"
        else: lead = f"{ppl_g - g:+d} gen" if (g is not None and ppl_g is not None) else "--"
        L.append(f"{disp} & {cell(g)} & {cell(l)} & {lead} \\\\")
    L += ["\\bottomrule", "\\end{tabular}", "\\end{table}"]; return "\n".join(L)

# ---------- Table 6: Self-BLEU vs Distinct-1 ----------
def latex_table6(metrics, label="selfbleu"):
    base = metrics[0]
    L = ["% Self-BLEU vs Distinct-1 (GPT-2)", "\\begin{table}[!t]",
         "\\caption{Self-BLEU vs. Distinct-1 comparison (GPT-2)}",
         f"\\label{{tab:{label}}}", "\\centering",
         "\\begin{tabular}{@{}ccccc@{}}", "\\toprule",
         "\\textbf{Gen} & \\textbf{Self-BLEU} & \\textbf{$\\Delta$SB} & \\textbf{Distinct-1} & \\textbf{$\\Delta$Distinct-1} \\\\", "\\midrule"]
    for g in sorted(metrics):
        if "self_bleu" not in metrics[g]: continue
        m = metrics[g]
        dsb = "--" if g == 0 else _fmt_pct((m["self_bleu"]/base["self_bleu"]-1)*100)
        dd1 = "--" if g == 0 else _fmt_pct((m["distinct_1"]/base["distinct_1"]-1)*100)
        L.append(f"{g} & {m['self_bleu']:.3f} & {dsb} & {m['distinct_1']:.3f} & {dd1} \\\\")
    L += ["\\bottomrule", "\\end{tabular}", "\\end{table}"]; return "\n".join(L)

# ---------- Table 7: threshold sensitivity ----------
def latex_table7(df, label="threshold_sensitivity"):
    L = ["% Threshold sensitivity (GPT-2)", "\\begin{table}[!t]",
         "\\caption{Threshold sensitivity: detection generation (GPT-2)}",
         f"\\label{{tab:{label}}}", "\\centering",
         "\\begin{tabular}{@{}cccc@{}}", "\\toprule",
         "\\textbf{Threshold} & \\textbf{Distinct-1 det.} & \\textbf{PPL det.} & \\textbf{Lead} \\\\", "\\midrule"]
    for _, r in df.iterrows():
        L.append(f"{r['Threshold']} & {r['Distinct-1 det.']} & {r['PPL det.']} & {r['Lead']} \\\\")
    L += ["\\bottomrule", "\\end{tabular}", "\\end{table}"]; return "\n".join(L)

# ---------- Table 8: mixed scenario ----------
def latex_table8(metrics, label="mixed_scenario"):
    base = metrics[0]
    L = ["% GPT-2 mixed scenario (30% synthetic)", "\\begin{table}[!t]",
         "\\caption{GPT-2: mixed scenario (30\\% synthetic)}",
         f"\\label{{tab:{label}}}", "\\centering",
         "\\begin{tabular}{@{}cccc@{}}", "\\toprule",
         "\\textbf{Gen} & \\textbf{Distinct-1} & \\textbf{$\\Delta$Distinct-1} & \\textbf{$\\Delta$PPL} \\\\", "\\midrule"]
    for g in sorted(metrics):
        m = metrics[g]
        dd1 = "--" if g == 0 else _fmt_pct((m["distinct_1"]/base["distinct_1"]-1)*100)
        dpp = "--" if g == 0 else _fmt_pct((m["perplexity"]/base["perplexity"]-1)*100)
        L.append(f"{g} & {m['distinct_1']:.3f} & {dd1} & {dpp} \\\\")
    L += ["\\bottomrule", "\\end{tabular}", "\\end{table}"]; return "\n".join(L)

# ---------- Table 9: text-quality degradation ----------
def latex_table9(metrics, samples, label="degradation", gens=(0, 3, 5)):
    L = ["% Text-quality degradation (GPT-2, 50 samples)", "\\begin{table}[!t]",
         "\\caption{Text-quality degradation (GPT-2, 50 samples)}",
         f"\\label{{tab:{label}}}", "\\centering",
         "\\begin{tabular}{@{}cccl@{}}", "\\toprule",
         "\\textbf{Gen} & \\textbf{Vocab size} & \\textbf{Repetition \\%} & \\textbf{Example} \\\\", "\\midrule"]
    for g in gens:
        if g not in metrics: continue
        m = metrics[g]
        ex = (samples.get(g, [""])[0] or "")[:34].replace("&", "\\&").replace("%", "\\%").replace("_", "\\_")
        L.append(f"{g} & {m['vocab_size']} & {m['repetition']*100:.0f}\\% & ``\\ldots {ex}\\ldots'' \\\\")
    L += ["\\bottomrule", "\\end{tabular}", "\\end{table}"]; return "\n".join(L)

def latex_comparison(det_llama, det_gpt2, label="comparison"):
    def lt(d):
        a, b = d.get("distinct_1"), d.get("perplexity")
        return f"{b-a:+d}" if (a is not None and b is not None) else "N/A"
    def c(x): return f"Gen {x}" if x is not None else "not det."
    return "\n".join([
        "% Detection timing comparison", "\\begin{table}[!t]",
        "\\caption{Detection Timing Comparison Across Architectures}",
        f"\\label{{tab:{label}}}", "\\centering",
        "\\begin{tabular}{@{}lccc@{}}", "\\toprule",
        "\\textbf{Model} & \\textbf{D-1 Detection} & \\textbf{PPL Detection} & \\textbf{Lead Time} \\\\", "\\midrule",
        f"LLaMA 3-8B & {c(det_llama['distinct_1'])} & {c(det_llama['perplexity'])} & {lt(det_llama)} gen \\\\",
        f"GPT-2 Medium & {c(det_gpt2['distinct_1'])} & {c(det_gpt2['perplexity'])} & {lt(det_gpt2)} gen \\\\",
        "\\bottomrule", "\\end{tabular}", "\\end{table}"])

tables = {
    "table1_lead_time.tex":   latex_table1(),
    "table2_framework.tex":   latex_table2(),
    "table3_llama3.tex":      latex_table3(llama_metrics),
    "table4_gpt2.tex":        latex_table4(gpt2_agg),
    "table5_detection.tex":   latex_table5(gpt2_det, llama_det),
    "table6_selfbleu.tex":    latex_table6(gpt2_metrics),
    "table7_threshold.tex":   latex_table7(gpt2_threshold_df),
    "table8_mixed.tex":       latex_table8(gpt2_mixed_metrics),
    "table9_degradation.tex": latex_table9(gpt2_metrics, gpt2_samples),
    "comparison_table.tex":   latex_comparison(llama_det, gpt2_det),
}
for fn, s in tables.items():
    with open(fn, "w") as f: f.write(s + "\n")
    print(f"  wrote {fn}")
print("\nAll paper tables (1-9 + comparison) regenerated from this run.")

In [ ]:
results = {
    "config": CONFIG,
    "timestamp": datetime.now().isoformat(),
    "llama3": {"scenario": "replace", "seed": CONFIG["seed"],
               "metrics": {str(k): v for k, v in llama_metrics.items()},
               "detection": llama_det},
    "gpt2": {"scenario": "replace", "seeds": CONFIG["seeds"],
             "metrics_mean": {str(k): v for k, v in gpt2_metrics.items()},
             "metrics_with_ci": {str(k): v for k, v in gpt2_agg.items()},
             "detection": gpt2_det},
    "gpt2_mixed": {"scenario": "mixed", "ratio": CONFIG["mixed_ratio"],
                   "metrics": {str(k): v for k, v in gpt2_mixed_metrics.items()}},
    "threshold_sweep_gpt2": gpt2_threshold_df.to_dict(orient="records"),
}
with open("experiment_results_complete.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved experiment_results_complete.json")

---
### Notes
- **Tables 1 & 2** are definitional framework tables and are emitted verbatim; **Tables 3–9, the comparison, and Figure 1 are computed from this run's metrics**.
- The **lead time** the run reports is whatever the live data shows (+1 or +2). If you want the published +2, use the calibration cell to confirm the ordering, and keep `num_generations`/`learning_rate_by_model` as tuned so collapse stays gradual enough to resolve the gap.
- `gpt2_samples` is now retained from seed 42, so Table 9 examples populate correctly.